# KOR KakaoTalk — Native vs Bizboard Impression Skew

| Field | Value |
|-------|-------|
| **Author** | Haewon |
| **Last update** | 2026-03-26 |
| **Objective** | Validate whether KakaoTalk sends ib and native bid requests at ~1:1 ratio but impressions skew toward Bizboard; identify root cause (VBT throttling vs creative compatibility) |
| **Scope** | KOR-targeting campaigns · Android + iOS · exchange = Kakao · formats: ib (Bizboard), ni/nl (native) |
| **Out of scope** | Non-Kakao exchanges · user quality analysis (covered in kr_vt_deepdive.ipynb Section 4) |
| **Key tables** | `moloco-ae-view.athena.fact_dsp_all`, `moloco-ae-view.athena.fact_dsp_publisher`, `focal-elf-631.prod_stream_view.imp` (VBT check, TBC) |
| **Additional context** | VBT = Value-Based Throttler: pre-pricing layer in bidfnt that drops low-value requests before bidding. Pipeline: bid_request → VBT → pricing → bid → win → impression |
| **Reference** | `Measurement/VT/kor_vt_deepdive_plan.md` — Section 5 |

In [34]:
import os
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [48]:
client = bigquery.Client(project="moloco-ods")

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    print(f'✅ {label}: {len(df)} rows')
    return df

CHART_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'charts'))
os.makedirs(CHART_DIR, exist_ok=True)

# ── Parameters ──────────────────────────────────────────────────────────────
DATE_START     = '2026-01-01'
DATE_END       = '2026-01-31'
TARGET_COUNTRY = 'KOR'
KAKAO_EXCHANGE = 'KAKAO'
FORMATS          = ['ib', 'ni', 'nl']  # ib = Bizboard, ni = native image, nl = native list
OS               = 'ANDROID'           # Android only for this analysis
KAKAOTALK_BUNDLE = 'com.kakao.talk'    # KakaoTalk publisher — same publisher across all sections

## Step 1 — Validate the 1:1 Bid Request Claim

**Goal:** Compare bid request volume for Bizboard (`ib`) vs native (`ni`/`nl`) from KakaoTalk exchange for KOR-targeting campaigns.  
**Expected output:** Ratio of Bizboard vs native bid requests — confirm whether it is ~1:1.

In [49]:
# Step 1: Single-source funnel from fact_supply
# fact_supply has bid_requests, bids, bids_won, impressions in one table — no cross-system mixing needed
# Canonical definitions:
#   bid_rate  = bids / bid_requests
#   win_rate  = bids_won / bids       (NOT impressions/bids — that's serve_rate)
#   serve_rate = impressions / bids_won
#
# cr_format breakdown: ib = Bizboard, ni = Native Image, nl = Native List
# Dimension notes:
#   exchange        → top-level, always UPPERCASE ('KAKAO')
#   req.app_bundle  → nested publisher app bundle
#   req.country     → 3-letter uppercase ('KOR')
#   req.os          → uppercase ('ANDROID')
#   date_utc        → partition column (mandatory filter)

query_funnel = f"""
SELECT
  cr_format,
  CASE cr_format
    WHEN 'ib' THEN 'Bizboard'
    WHEN 'ni' THEN 'Native Image'
    WHEN 'nl' THEN 'Native List'
    ELSE cr_format
  END                   AS format_label,
  SUM(bid_requests)     AS bid_requests,
  SUM(bids)             AS bids,
  SUM(bids_won)         AS bids_won,
  SUM(impressions)      AS impressions,
  SUM(media_cost_usd)   AS spend_usd
FROM `moloco-ae-view.athena.fact_supply`
WHERE date_utc  BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND exchange        = '{KAKAO_EXCHANGE}'
  AND req.app_bundle  = '{KAKAOTALK_BUNDLE}'
  AND req.country     = '{TARGET_COUNTRY}'
  AND req.os          = 'ANDROID'
  AND cr_format IN ('ib', 'ni', 'nl')
GROUP BY 1, 2
ORDER BY 1
"""

df_raw = run_query(query_funnel, label='Funnel from fact_supply — ib / ni / nl')
df_raw


✅ Funnel from fact_supply — ib / ni / nl: 3 rows


,cr_format,format_label,bid_requests,bids,bids_won,impressions,spend_usd
0,ib,Bizboard,28387030000,3983199500,2043768900,1722205286,591679.467126
1,ni,Native Image,8757240000,229422100,31053500,5009602,14494.179536
2,nl,Native List,0,4703467100,330184000,211431112,22720.002576


In [50]:
# Compute canonical rates per format (ib / ni / nl) + combined Native group
df_funnel = df_raw.copy()
df_funnel['bid_rate']   = df_funnel['bids']       / df_funnel['bid_requests'].replace(0, float('nan'))
df_funnel['win_rate']   = df_funnel['bids_won']    / df_funnel['bids'].replace(0, float('nan'))
df_funnel['serve_rate'] = df_funnel['impressions'] / df_funnel['bids_won'].replace(0, float('nan'))

# Build combined Native row (ni + nl)
nat_cols = ['bid_requests', 'bids', 'bids_won', 'impressions', 'spend_usd']
nat_row  = df_funnel[df_funnel['cr_format'].isin(['ni', 'nl'])][nat_cols].sum()
nat_row['cr_format']    = 'nat'
nat_row['format_label'] = 'Native (total)'
nat_row['bid_rate']     = nat_row['bids']       / nat_row['bid_requests'] if nat_row['bid_requests'] else float('nan')
nat_row['win_rate']     = nat_row['bids_won']   / nat_row['bids']         if nat_row['bids']         else float('nan')
nat_row['serve_rate']   = nat_row['impressions']/ nat_row['bids_won']     if nat_row['bids_won']     else float('nan')

# df_funnel_full = ib / ni / nl / Native(total)
df_funnel_full = pd.concat([df_funnel, nat_row.to_frame().T], ignore_index=True)
df_funnel_full[nat_cols] = df_funnel_full[nat_cols].apply(pd.to_numeric)

print("Funnel summary (Android · KakaoTalk · KOR):")
for metric in ['bid_requests', 'bids', 'bids_won', 'impressions']:
    vals  = {row['format_label']: row[metric] for _, row in df_funnel_full.iterrows()}
    ib    = vals.get('Bizboard', 0)
    ni    = vals.get('Native Image', 0)
    nl    = vals.get('Native List', 0)
    nat   = vals.get('Native (total)', 0)
    ratio = ib / nat if nat > 0 else float('nan')
    print(f"  {metric:15s}: ib={ib:>12,.0f}  ni={ni:>12,.0f}  nl={nl:>12,.0f}  native={nat:>12,.0f}  ib:native={ratio:.2f}x")

print()
rate_cols = ['format_label', 'bid_requests', 'bids', 'bids_won', 'impressions', 'bid_rate', 'win_rate', 'serve_rate']
fmt_out = df_funnel_full[rate_cols].copy()
for c in ['bid_rate', 'win_rate', 'serve_rate']:
    fmt_out[c] = fmt_out[c].map(lambda v: f'{v:.2%}' if pd.notna(v) else '—')
print(fmt_out.to_string(index=False))


Funnel summary (Android · KakaoTalk · KOR):
  bid_requests   : ib=28,387,030,000  ni=8,757,240,000  nl=           0  native=8,757,240,000  ib:native=3.24x
  bids           : ib=3,983,199,500  ni= 229,422,100  nl=4,703,467,100  native=4,932,889,200  ib:native=0.81x
  bids_won       : ib=2,043,768,900  ni=  31,053,500  nl= 330,184,000  native= 361,237,500  ib:native=5.66x
  impressions    : ib=1,722,205,286  ni=   5,009,602  nl= 211,431,112  native= 216,440,714  ib:native=7.96x

  format_label  bid_requests       bids   bids_won  impressions bid_rate win_rate serve_rate
      Bizboard   28387030000 3983199500 2043768900   1722205286   14.03%   51.31%     84.27%
  Native Image    8757240000  229422100   31053500      5009602    2.62%   13.54%     16.13%
   Native List             0 4703467100  330184000    211431112        —    7.02%     64.03%
Native (total)    8757240000 4932889200  361237500    216440714   56.33%    7.32%     59.92%


## Step 2 — Quantify the Impression Skew & Funnel Breakdown

**Goal:** Identify at which funnel stage the Bizboard vs native divergence occurs.

| Divergence stage | Interpretation |
|-----------------|----------------|
| `bid_requests → bids` (bid_rate) | VBT throttling — pre-pricing, no campaign evaluated |
| `bids → bids_won` (win_rate) | Auction loss — bid submitted but lost exchange auction |
| `bids_won → impressions` (serve_rate) | Render/serve issue — won but not rendered |

Canonical definitions (from `fact_supply`):
- **bid_rate** = bids / bid_requests
- **win_rate** = bids_won / bids
- **serve_rate** = impressions / bids_won


In [51]:
# Step 2: Funnel viz — bid_requests → bids → bids_won → impressions
# Shows ib / ni / nl individually + Native (total) as combined group

stages = ['bid_requests', 'bids', 'bids_won', 'impressions']
labels = ['Bid Requests', 'Bids', 'Bids Won', 'Impressions']
colors = {
    'Bizboard':       '#636EFA',
    'Native Image':   '#EF553B',
    'Native List':    '#FF9966',
    'Native (total)': '#B22222',   # darker red — summary group
}

fig_funnel = go.Figure()
for _, row in df_funnel_full.iterrows():
    is_total = row['format_label'] == 'Native (total)'
    fig_funnel.add_trace(go.Bar(
        name=row['format_label'],
        x=labels,
        y=[row[s] for s in stages],
        marker_color=colors.get(row['format_label']),
        marker_pattern_shape='/' if is_total else '',   # hatched for summary
        opacity=0.7 if is_total else 1.0,
        text=[f'{row[s]/1e6:.2f}M' if row[s] >= 1e5 else f'{row[s]:,.0f}' for s in stages],
        textposition='outside'
    ))

fig_funnel.update_layout(
    barmode='group',
    title=f'Funnel: ib / ni / nl / Native(total) — KakaoTalk · Android · {TARGET_COUNTRY} · {DATE_START}–{DATE_END}',
    yaxis_title='Volume',
    legend_title='Format',
    height=480
)
fig_funnel.show()
fig_funnel.write_image(os.path.join(CHART_DIR, f's1_funnel_kakao_ib_ni_nl_ANDROID.png'), scale=2)
print('Saved: s1_funnel_kakao_ib_ni_nl_ANDROID.png')

# Rate summary table
print("\nFunnel rate summary:")
rate_cols = ['format_label', 'bid_requests', 'bids', 'bids_won', 'impressions', 'bid_rate', 'win_rate', 'serve_rate']
fmt_out = df_funnel_full[rate_cols].copy()
for c in ['bid_rate', 'win_rate', 'serve_rate']:
    fmt_out[c] = fmt_out[c].map(lambda v: f'{v:.2%}' if pd.notna(v) else '—')
print(fmt_out.to_string(index=False))


Saved: s1_funnel_kakao_ib_ni_nl_ANDROID.png

Funnel rate summary:
  format_label  bid_requests       bids   bids_won  impressions bid_rate win_rate serve_rate
      Bizboard   28387030000 3983199500 2043768900   1722205286   14.03%   51.31%     84.27%
  Native Image    8757240000  229422100   31053500      5009602    2.62%   13.54%     16.13%
   Native List             0 4703467100  330184000    211431112        —    7.02%     64.03%
Native (total)    8757240000 4932889200  361237500    216440714   56.33%    7.32%     59.92%


## Step 3 — Funnel Deep Dive: bid_result Breakdown by Format × Publisher

**Source:** `focal-elf-631.prod.trace{YYYYMMDD}` — actual production bid decisions with all funnel stage codes.

Key `bid_result` values:

| Value | Stage |
|---|---|
| `BID` | Bid placed ✅ |
| `THROTTLED_SHORT_TERM_USER_PROFILE` | VBT throttled |
| `THROTTLED_BID_CAP` | Bid cap throttled |
| `THROTTLED_PROBABILISTICALLY` | Probabilistic throttle |
| `NO_WINNER` | Internal auction — no winner |
| `TIME_OUT` | Timed out |
| others | Other filters |

**Filter:** `exchange = 'KAKAO'` + `app_bundle = 'com.kakao.talk'` — **within the same publisher** to isolate format as the variable.  
⚠️ Table is ~1.14 TB/day — using a **7-day window** (`DATE_END - 6d → DATE_END`) to control cost.

> **Note on ni vs nl:** `prod.trace` only has `inlentory_format` (`B` = Bizboard, `N` = Native) — it does **not** distinguish Native Image (`ni`) from Native Video (`nl`). For ni/nl breakdown use Steps 1/2 (`fact_supply`). Step 3 shows Bizboard vs Native (combined).


In [52]:
# Step 3: bid_result breakdown from prod.trace (canonical approach)
# Reference: Bidfnt/bidding_funnel_analysis.sql
# - prod.trace has ONE ROW PER BID REQUEST (bid_result = overall request outcome)
# - Use ROUND(SUM(1/rate)) for extrapolated volumes (rate = sampling rate per row)
# - cr_format in trace is only set for BID rows (post creative selection) — NULL for THROTTLED rows
# - Must use inventory_format (B/N) which is set at request level for ALL rows incl. throttled
# - prod.trace has explicit 'os' column
# Scope: exchange=KAKAO, app_bundle=KakaoTalk, os=ANDROID, country=KOR — same as Steps 1/2

from datetime import datetime, timedelta

_trace_start = (datetime.strptime(DATE_END, '%Y-%m-%d') - timedelta(days=6)).strftime('%Y-%m-%d')
_yr       = DATE_END[:4]
_s_sfx_tr = _trace_start[5:7] + _trace_start[8:10]
_e_sfx_tr = DATE_END[5:7]     + DATE_END[8:10]
print(f"Trace window: {_trace_start} → {DATE_END}")

# (A) Request-level rate summary
query_trace_rates = f"""
SELECT
  CASE inventory_format
    WHEN 'B' THEN 'Bizboard (ib)'
    WHEN 'N' THEN 'Native (ni+nl)'
    ELSE inventory_format
  END                                                   AS format_group,
  ROUND(SUM(CASE WHEN bid_result = 'BID'
                 THEN 1/rate ELSE 0 END))               AS bid_vol,
  ROUND(SUM(CASE WHEN bid_result LIKE 'THROTTLED%'
                 THEN 1/rate ELSE 0 END))               AS throttled_vol,
  ROUND(SUM(CASE WHEN bid_result = 'THROTTLED_SHORT_TERM_USER_PROFILE'
                 THEN 1/rate ELSE 0 END))               AS vbt_vol,
  ROUND(SUM(1/rate))                                    AS total_vol,
  SAFE_DIVIDE(
    SUM(CASE WHEN bid_result = 'BID' THEN 1/rate ELSE 0 END),
    SUM(1/rate))                                        AS bid_rate,
  SAFE_DIVIDE(
    SUM(CASE WHEN bid_result LIKE 'THROTTLED%' THEN 1/rate ELSE 0 END),
    SUM(1/rate))                                        AS throttle_rate,
  SAFE_DIVIDE(
    SUM(CASE WHEN bid_result = 'THROTTLED_SHORT_TERM_USER_PROFILE' THEN 1/rate ELSE 0 END),
    SUM(1/rate))                                        AS vbt_rate
FROM `focal-elf-631.prod.trace{_yr}*`
WHERE _TABLE_SUFFIX BETWEEN '{_s_sfx_tr}' AND '{_e_sfx_tr}'
  AND DATE(timestamp)  BETWEEN '{_trace_start}' AND '{DATE_END}'
  AND exchange         = '{KAKAO_EXCHANGE}'
  AND app_bundle       = '{KAKAOTALK_BUNDLE}'
  AND UPPER(os)        = '{OS}'
  AND country          = '{TARGET_COUNTRY}'
  AND inventory_format IN ('B', 'N')
GROUP BY 1
"""

# (B) bid_result breakdown for stacked bar
query_trace_eval = f"""
SELECT
  CASE inventory_format
    WHEN 'B' THEN 'Bizboard (ib)'
    WHEN 'N' THEN 'Native (ni+nl)'
    ELSE inventory_format
  END                                AS format_group,
  bid_result,
  ROUND(SUM(1/rate))                 AS requests_est
FROM `focal-elf-631.prod.trace{_yr}*`
WHERE _TABLE_SUFFIX BETWEEN '{_s_sfx_tr}' AND '{_e_sfx_tr}'
  AND DATE(timestamp)  BETWEEN '{_trace_start}' AND '{DATE_END}'
  AND exchange         = '{KAKAO_EXCHANGE}'
  AND app_bundle       = '{KAKAOTALK_BUNDLE}'
  AND UPPER(os)        = '{OS}'
  AND country          = '{TARGET_COUNTRY}'
  AND inventory_format IN ('B', 'N')
GROUP BY 1, 2
ORDER BY format_group, requests_est DESC
"""

df_trace_rates = run_query(query_trace_rates, label='Request-level bid rates (prod.trace)')
df_trace_eval  = run_query(query_trace_eval,  label='bid_result breakdown (prod.trace)')
df_trace_rates


Trace window: 2026-01-25 → 2026-01-31
✅ Request-level bid rates (prod.trace): 2 rows
✅ bid_result breakdown (prod.trace): 33 rows


,format_group,bid_vol,throttled_vol,vbt_vol,total_vol,bid_rate,throttle_rate,vbt_rate
0,Bizboard (ib),858300000.0,2.584500e+09,2.279900e+09,6.262605e+09,0.137052,0.412688,0.364050
1,Native (ni+nl),968700000.0,8.376000e+08,6.603000e+08,2.015102e+09,0.480720,0.415661,0.327676


In [53]:
# Rate summary (request-level, rate-corrected — comparable to Step 1/2)
print("Bid rate & throttle rate summary — ib / ni / nl (Android · KakaoTalk · KOR):")
cols = ['format_group', 'total_vol', 'bid_rate', 'throttle_rate', 'vbt_rate']
fmt = df_trace_rates[cols].copy()
fmt['total_vol'] = fmt['total_vol'].map(lambda v: f'{v:,.0f}')
for c in ['bid_rate', 'throttle_rate', 'vbt_rate']:
    fmt[c] = fmt[c].map(lambda v: f'{v:.1%}' if pd.notna(v) else '—')
print(fmt.to_string(index=False))

# Full bid_result reason map (from Bidfnt/bidding_funnel_analysis.sql reference)
REASON_MAP = {
    'BID':                                          'BID (placed)',
    'NO_WINNER':                                    'No winner',
    'NO_FINALIST':                                  'No finalist',
    'NO_CAMPAIGN_AFTER_REQ_FILTER':                 'No campaign (req filter)',
    'NO_CAMPAIGN_AFTER_CTX_FILTER':                 'No campaign (ctx filter)',
    'NO_ADGROUP_AFTER_CTX_FILTER':                  'No ad group',
    'FILTERED_INCOMPATIBLE_TARGET_INVENTORY':       'Filtered: incompat. inlentory',  # creative compat
    'FILTERED_PUBLISHER':                           'Filtered: publisher',
    'FILTERED_UNPUBLISHED_APP':                     'Filtered: unpublished app',
    'TIME_OUT':                                     'Timeout',
    'THROTTLED_SHORT_TERM_USER_PROFILE':            'Throttled: VBT',
    'THROTTLED_PROBABILISTICALLY':                  'Throttled: probabilistic',
    'THROTTLED_BID_CAP':                            'Throttled: bid interval capper',
    'THROTTLED_BOOTUP':                             'Throttled: bootup',
    'THROTTLED_BY_EXCHANGE_INCREMENTALITY_EXP':     'Throttled: incr. exp.',
    'THROTTLED_BIDFLOOR_THROTTLER':                 'Throttled: bid floor',
    'THROTTLED_UPT':                                'Throttled: UPT',
    'THROTTLED_POSTPRICING_WIN_PRED':               'Throttled: post-pricing',
}
REASON_COLORS = {
    'BID (placed)':                   '#2CA02C',
    'No winner':                      '#AEC7E8',
    'No finalist':                    '#C5B0D5',
    'No campaign (req filter)':       '#9467BD',
    'No campaign (ctx filter)':       '#C49C94',
    'No ad group':                    '#E377C2',
    'Filtered: incompat. inlentory':  '#17BECF',  # highlight — creative compat signal
    'Filtered: publisher':            '#BCBD22',
    'Filtered: unpublished app':      '#7F7F7F',
    'Timeout':                        '#FFBB78',
    'Throttled: VBT':                 '#D62728',
    'Throttled: probabilistic':       '#EF553B',
    'Throttled: bid interval capper': '#FF7F0E',
    'Throttled: bootup':              '#8C564B',
    'Throttled: incr. exp.':          '#F7B6D2',
    'Throttled: bid floor':           '#DBDB8D',
    'Throttled: UPT':                 '#9EDAE5',
    'Throttled: post-pricing':        '#AAFFC3',
    'Other':                          '#C7C7C7',
}
REASON_ORDER = list(REASON_COLORS.keys())

df_trace_eval['reason_label'] = df_trace_eval['bid_result'].map(REASON_MAP).fillna('Other')
df_trace_eval['pct'] = df_trace_eval.groupby('format_group')['requests_est'].transform(
    lambda x: x / x.sum() * 100
)

d = df_trace_eval.groupby(['format_group', 'reason_label'])['pct'].sum().reset_index()

fig_trace = go.Figure()
for reason in REASON_ORDER:
    rd = d[d['reason_label'] == reason]
    if rd.empty:
        continue
    fig_trace.add_trace(go.Bar(
        name=reason,
        x=rd['format_group'],
        y=rd['pct'],
        marker_color=REASON_COLORS.get(reason, '#C7C7C7'),
        text=rd['pct'].map(lambda v: f'{v:.1f}%' if v >= 1 else ''),
        textposition='inside'
    ))

fig_trace.update_layout(
    barmode='stack',
    title=f'Bid Result Breakdown — KakaoTalk · Android · {TARGET_COUNTRY} · {_trace_start}–{DATE_END}',
    yaxis_title='% of bid requests',
    yaxis_ticksuffix='%',
    legend_title='Bid Result',
    height=500
)
fig_trace.show()
fig_trace.write_image(os.path.join(CHART_DIR, f's3_bid_result_breakdown_kakao_ANDROID.png'), scale=2)
print('Saved: s3_bid_result_breakdown_kakao_ANDROID.png')
print("\nNote: 'Filtered: incompat. inlentory' = FILTERED_INCOMPATIBLE_TARGET_INVENTORY — key creative compatibility signal")

Bid rate & throttle rate summary — ib / ni / nl (Android · KakaoTalk · KOR):
  format_group     total_vol bid_rate throttle_rate vbt_rate
 Bizboard (ib) 6,262,604,785    13.7%         41.3%    36.4%
Native (ni+nl) 2,015,102,374    48.1%         41.6%    32.8%


Saved: s3_bid_result_breakdown_kakao_ANDROID.png

Note: 'Filtered: incompat. inlentory' = FILTERED_INCOMPATIBLE_TARGET_INVENTORY — key creative compatibility signal


## Step 3b — Win Rate Deep Dive: Bid Floor Comparison

**Goal:** Explain the win rate gap (bids → impressions). Once Moloco places a bid, why does Bizboard win more than Native?  
**Hypothesis:** Native bid floors from KakaoTalk are higher → Moloco's bid is below floor more often → lower win rate.  
**Source:** `focal-elf-631.prod.bidrequest{YYYY}*` — bid floor (`bidfloor`) by format from incoming bid requests.

In [54]:
# Step 3b-2: Bid floor distribution by format from sampled bid request table
# Higher bid floors on native → Moloco underbids → lower win rate
# Note: bidfloor in the raw table is in the exchange currency (usually USD)
# _yr / _s_sfx / _e_sfx derived here (fact_supply is used in Step 1 so these are not set elsewhere)

_yr    = DATE_START[:4]
_s_sfx = DATE_START[5:7] + DATE_START[8:10]
_e_sfx = DATE_END[5:7]   + DATE_END[8:10]

query_bidfloor = f"""
SELECT
  CASE inventory_format
    WHEN 'B' THEN 'Bizboard (Banner)'
    WHEN 'N' THEN 'Native'
    ELSE inventory_format
  END                                           AS format_group,
  UPPER(os)                                     AS os,
  APPROX_QUANTILES(bidfloor, 100)[OFFSET(25)]  AS p25_bidfloor,
  APPROX_QUANTILES(bidfloor, 100)[OFFSET(50)]  AS p50_bidfloor,
  APPROX_QUANTILES(bidfloor, 100)[OFFSET(75)]  AS p75_bidfloor,
  APPROX_QUANTILES(bidfloor, 100)[OFFSET(90)]  AS p90_bidfloor,
  AVG(bidfloor)                                 AS avg_bidfloor,
  COUNT(*)                                      AS sampled_requests
FROM `focal-elf-631.prod.bidrequest{_yr}*`
WHERE _TABLE_SUFFIX BETWEEN '{_s_sfx}' AND '{_e_sfx}'
  AND DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND exchange        = '{KAKAO_EXCHANGE}'
  AND country         = '{TARGET_COUNTRY}'
  AND UPPER(os)       = 'ANDROID'
  AND inventory_format IN ('B', 'N')
  AND app_bundle      = '{KAKAOTALK_BUNDLE}'
  AND bidfloor        > 0
GROUP BY 1, 2
ORDER BY format_group, os
"""

df_floor = run_query(query_bidfloor, label='Bid floor distribution by format (Kakao)')
df_floor


✅ Bid floor distribution by format (Kakao): 2 rows


,format_group,os,p25_bidfloor,p50_bidfloor,p75_bidfloor,p90_bidfloor,avg_bidfloor,sampled_requests
0,Bizboard (Banner),ANDROID,0.204158,0.206697,0.207917,0.208296,0.208005,2838704
1,Native,ANDROID,0.006860,0.006931,0.068623,0.203569,0.044668,875733


In [55]:
# Visualise bid floor percentiles side-by-side — Bizboard vs Native per OS

os_val = 'ANDROID'
if True:
    d = df_floor[df_floor['os'] == os_val]
    pcts    = ['p25_bidfloor', 'p50_bidfloor', 'p75_bidfloor', 'p90_bidfloor']
    labels  = ['P25', 'P50 (median)', 'P75', 'P90']
    colors  = {'Bizboard (ib)': '#636EFA', 'Native': '#EF553B'}

    fig_fl = go.Figure()
    for _, row in d.iterrows():
        fig_fl.add_trace(go.Bar(
            name=row['format_group'],
            x=labels,
            y=[row[p] for p in pcts],
            marker_color=colors.get(row['format_group']),
            text=[f'${row[p]:.3f}' for p in pcts],
            textposition='outside'
        ))

    fig_fl.update_layout(
        barmode='group',
        title=f'Bid Floor Distribution — Bizboard vs Native | Kakao · {os_val} · {TARGET_COUNTRY} · {DATE_START}–{DATE_END}',
        yaxis_title='Bid floor (USD CPM)',
        legend_title='Format',
        height=420
    )
    fig_fl.show()
    fig_fl.write_image(os.path.join(CHART_DIR, f's3b_bidfloor_kakao_{os_val}.png'), scale=2)
    print(f'Saved: s3b_bidfloor_kakao_{os_val}.png')

print("\nBid floor summary:")
print(df_floor[['format_group', 'os', 'p50_bidfloor', 'p75_bidfloor', 'p90_bidfloor', 'avg_bidfloor']].to_string(index=False))

Saved: s3b_bidfloor_kakao_ANDROID.png

Bid floor summary:
     format_group      os  p50_bidfloor  p75_bidfloor  p90_bidfloor  avg_bidfloor
Bizboard (Banner) ANDROID      0.206697      0.207917      0.208296      0.208005
           Native ANDROID      0.006931      0.068623      0.203569      0.044668


## Step 4 — Bid Price Comparison (ib vs ni/nl)

**Goal:** Determine whether Moloco underbids on native relative to Bizboard, explaining the win rate gap (51% ib vs 14% native).

**Source:** `focal-elf-631.prod_stream_view.pricing` — internal auction data, 1/1000 sample.  
`candidates.candidate_result = 'CommitBid'` filters to actual bids submitted to the exchange.

| Metric | Description |
|--------|-------------|
| `bid_price` | Moloco's bid submitted to Kakao exchange |
| `bid_floor` | Exchange floor price from bid request |
| `bid_price / bid_floor` | Coverage ratio — how much above floor Moloco bids |

In [56]:
# Step 4-A: Bid price distribution by format from prod_stream_view.pricing
# 1/1000 sample — multiply COUNT(*) by 1000 for estimated volume
# CommitBid = actual bid submitted to exchange (internal auction winner)
# Schema confirmed:
#   candidates.bid_price              = INTEGER (micro-CPM) — divide by 1e6 for CPM in USD
#   req.imp.bidfloor                  = STRUCT<currency STRING, amount_micro INT64>
#   req.imp.bidfloor.amount_micro     = INTEGER (micro units) — divide by 1e6 for bid floor in USD
# Use CTE to extract scalars before aggregating (avoids BQ STRUCT-in-aggregate limitation).

query_bid_price = f"""
WITH flat AS (
  SELECT
    CASE candidates.cr_format
      WHEN 'ib' THEN 'Bizboard (ib)'
      WHEN 'ni' THEN 'Native Image'
      WHEN 'nl' THEN 'Native List'
      ELSE candidates.cr_format
    END                                       AS format_label,
    candidates.cr_format                      AS cr_format,
    req.imp.bidfloor.amount_micro / 1e6       AS bidfloor_usd,
    candidates.bid_price / 1e6                AS bid_price_usd
  FROM `focal-elf-631.prod_stream_view.pricing`,
    UNNEST(pricing.candidates) AS candidates
  WHERE DATE(timestamp) BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND req.exchange   = '{KAKAO_EXCHANGE}'
    AND req.app.bundle = '{KAKAOTALK_BUNDLE}'
    AND UPPER(os)      = '{OS}'
    AND country        = '{TARGET_COUNTRY}'
    AND candidates.cr_format IN ('ib', 'ni', 'nl')
    AND candidates.candidate_result = 'CommitBid'
)
SELECT
  format_label,
  cr_format,
  COUNT(*) * 1000                                         AS est_bids,
  AVG(bidfloor_usd)                                       AS avg_bid_floor,
  AVG(bid_price_usd)                                      AS avg_bid_price,
  APPROX_QUANTILES(bid_price_usd, 100)[OFFSET(25)]         AS p25_bid_price,
  APPROX_QUANTILES(bid_price_usd, 100)[OFFSET(50)]         AS p50_bid_price,
  APPROX_QUANTILES(bid_price_usd, 100)[OFFSET(75)]         AS p75_bid_price,
  APPROX_QUANTILES(bid_price_usd, 100)[OFFSET(90)]         AS p90_bid_price,
  SAFE_DIVIDE(
    AVG(bid_price_usd), NULLIF(AVG(bidfloor_usd), 0)
  )                                                        AS bid_to_floor_ratio
FROM flat
GROUP BY 1, 2
ORDER BY cr_format
"""

df_bid_price = run_query(query_bid_price, label='Bid price distribution by format (pricing table)')

fmt_bp = df_bid_price.copy()
for c in ['avg_bid_floor', 'avg_bid_price', 'p25_bid_price', 'p50_bid_price', 'p75_bid_price', 'p90_bid_price']:
    fmt_bp[c] = fmt_bp[c].map(lambda v: f'${v:.4f}' if pd.notna(v) else '—')
fmt_bp['bid_to_floor_ratio'] = fmt_bp['bid_to_floor_ratio'].map(lambda v: f'{v:.2f}x' if pd.notna(v) else '—')
fmt_bp['est_bids'] = fmt_bp['est_bids'].map(lambda v: f'{v:,.0f}')
print(fmt_bp.to_string(index=False))


✅ Bid price distribution by format (pricing table): 3 rows
 format_label cr_format      est_bids avg_bid_floor avg_bid_price p25_bid_price p50_bid_price p75_bid_price p90_bid_price bid_to_floor_ratio
Bizboard (ib)        ib 3,974,227,000       $0.0002       $0.0003       $0.0002       $0.0002       $0.0003       $0.0005              1.62x
 Native Image        ni   228,113,000       $0.0002       $0.0009       $0.0002       $0.0004       $0.0009       $0.0020              4.44x
  Native List        nl 4,697,141,000       $0.0000       $0.0001       $0.0000       $0.0001       $0.0001       $0.0002              3.76x


In [57]:
# Step 4-B: Bid price distribution chart — P25/P50/P75/P90 + avg bid floor
colors = {'Bizboard (ib)': '#636EFA', 'Native Image': '#EF553B', 'Native List': '#FF9966'}

fig_bp = go.Figure()

for _, row in df_bid_price.iterrows():
    fmt   = row['format_label']
    color = colors.get(fmt, '#888')
    x     = [fmt]

    # P25–P75 bar
    fig_bp.add_trace(go.Bar(
        name=fmt, x=x,
        y=[row['p75_bid_price'] - row['p25_bid_price']],
        base=[row['p25_bid_price']],
        marker_color=color, opacity=0.5, width=0.35, showlegend=True
    ))
    # P50 marker
    fig_bp.add_trace(go.Scatter(
        x=x, y=[row['p50_bid_price']], mode='markers',
        marker=dict(symbol='line-ew', size=22, color=color, line=dict(width=3, color=color)),
        showlegend=False
    ))
    # Avg bid price
    fig_bp.add_trace(go.Scatter(
        x=x, y=[row['avg_bid_price']], mode='markers',
        marker=dict(symbol='diamond', size=10, color=color),
        showlegend=False
    ))
    # Avg bid floor
    fig_bp.add_trace(go.Scatter(
        x=x, y=[row['avg_bid_floor']], mode='markers',
        marker=dict(symbol='x', size=12, color='black', line=dict(width=2)),
        showlegend=False
    ))

fig_bp.update_layout(
    barmode='overlay',
    title=(f'Bid Price Distribution — KakaoTalk · Android · {TARGET_COUNTRY} · {DATE_START}–{DATE_END}<br>'
           '<sup>Bar = P25–P75 | line = P50 | diamond = avg bid price | x = avg bid floor</sup>'),
    yaxis_title='Price (USD)',
    height=480
)
fig_bp.show()
fig_bp.write_image(os.path.join(CHART_DIR, 's4_bid_price_distribution_ANDROID.png'), scale=2)
print('Saved: s4_bid_price_distribution_ANDROID.png')

print('\nBid-to-floor ratio summary:')
for _, row in df_bid_price.iterrows():
    print(f"  {row['format_label']:20s}  avg_bid=${row['avg_bid_price']:.4f}  "
          f"avg_floor=${row['avg_bid_floor']:.4f}  ratio={row['bid_to_floor_ratio']:.2f}x")


Saved: s4_bid_price_distribution_ANDROID.png

Bid-to-floor ratio summary:
  Bizboard (ib)         avg_bid=$0.0003  avg_floor=$0.0002  ratio=1.62x
  Native Image          avg_bid=$0.0009  avg_floor=$0.0002  ratio=4.44x
  Native List           avg_bid=$0.0001  avg_floor=$0.0000  ratio=3.76x


## Appendix — Bid Rate Discrepancy: fact_supply vs prod.trace

**Hypothesis:** `prod.trace inventory_format='N'` captures a broader set of creative formats than `fact_supply cr_format IN ('ni','nl')`, inflating the trace-side native bid rate.

**Investigation:** Cross-tab `inventory_format × cr_format` in `fact_supply` to see which cr_formats map to Native inventory, and compare bid rates per cr_format.

In [58]:
# A: Cross-tab inventory_format × cr_format in fact_supply
# Shows which cr_formats sit under inventory_format='Native' and their individual bid rates

query_format_xref = f"""
SELECT
  inventory_format,
  cr_format,
  SUM(bid_requests)   AS bid_requests,
  SUM(bids)           AS bids,
  SUM(impressions)    AS impressions,
  SAFE_DIVIDE(SUM(bids), NULLIF(SUM(bid_requests), 0))   AS bid_rate,
  SAFE_DIVIDE(SUM(impressions), NULLIF(SUM(bids), 0))    AS win_rate
FROM `moloco-ae-view.athena.fact_supply`
WHERE date_utc     BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND exchange        = '{KAKAO_EXCHANGE}'
  AND req.app_bundle  = '{KAKAOTALK_BUNDLE}'
  AND req.country     = '{TARGET_COUNTRY}'
  AND req.os          = 'ANDROID'
GROUP BY 1, 2
ORDER BY inventory_format, bid_requests DESC
"""

df_xref = run_query(query_format_xref, label='inventory_format x cr_format cross-tab')

fmt_xref = df_xref.copy()
for c in ['bid_rate', 'win_rate']:
    fmt_xref[c] = fmt_xref[c].map(lambda v: f'{v:.1%}' if pd.notna(v) else '—')
for c in ['bid_requests', 'bids', 'impressions']:
    fmt_xref[c] = fmt_xref[c].map(lambda v: f'{v:,.0f}')
print(fmt_xref.to_string(index=False))


✅ inventory_format x cr_format cross-tab: 3 rows
inventory_format cr_format   bid_requests          bids   impressions bid_rate win_rate
          Banner        ib 28,387,030,000 3,983,199,500 1,722,205,286    14.0%    43.2%
          Native        ni  8,757,240,000   229,422,100     5,009,602     2.6%     2.2%
          Native        nl              0 4,703,467,100   211,431,112        —     4.5%


In [59]:
# B: Reconcile volumes — fact_supply grouped by inventory_format vs prod.trace total_vol
# Normalise prod.trace (7d) to full-month equivalent (x31/7) before comparing

df_by_inlfmt = df_xref.groupby('inventory_format').agg(
    bid_requests=('bid_requests', 'sum'),
    bids=('bids', 'sum'),
    impressions=('impressions', 'sum')
).reset_index()
df_by_inlfmt['bid_rate'] = df_by_inlfmt['bids'] / df_by_inlfmt['bid_requests'].replace(0, float('nan'))

tr_map = df_trace_rates.set_index('format_group')['total_vol'].to_dict()
tr_bid = df_trace_rates.set_index('format_group')['bid_vol'].to_dict()

# inventory_format label → trace format_group
inl_to_trace = {'Banner': 'Bizboard (ib)', 'Native': 'Native'}

print("Reconciliation: fact_supply (by inventory_format) vs prod.trace (normalised to 31d)")
print(f"{'inventory_format':22s} {'fs_req':>16s} {'tr_req(31d)':>16s} {'req_match':>10s} {'fs_bid_rate':>12s} {'tr_bid_rate':>12s}")
print("-" * 92)
for _, row in df_by_inlfmt.iterrows():
    inl    = row['inventory_format']
    tr_key = inl_to_trace.get(inl)
    tr_req_7d = tr_map.get(tr_key, float('nan'))
    tr_req_31 = tr_req_7d * 31 / 7 if tr_key else float('nan')
    tr_br     = (tr_bid.get(tr_key, float('nan')) / tr_req_7d) if tr_key and tr_req_7d else float('nan')
    match     = row['bid_requests'] / tr_req_31 if tr_req_31 else float('nan')
    print(f"{inl:22s} {row['bid_requests']:>16,.0f} {tr_req_31:>16,.0f} {match:>10.1%} {row['bid_rate']:>12.1%} {tr_br:>12.1%}")


Reconciliation: fact_supply (by inventory_format) vs prod.trace (normalised to 31d)
inventory_format                 fs_req      tr_req(31d)  req_match  fs_bid_rate  tr_bid_rate
--------------------------------------------------------------------------------------------
Banner                   28,387,030,000   27,734,392,619     102.4%        14.0%        13.7%
Native                    8,757,240,000              nan       nan%        56.3%         nan%


## Export — Save Results to CSV

In [60]:
# Export all step results to CSV
# Saved to Measurement/VT/data/

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
os.makedirs(DATA_DIR, exist_ok=True)

exports = {
    'kakao_s1_funnel_fact_supply.csv':           df_funnel,        # Step 1: bid_requests/bids/bids_won/imp by format
    'kakao_s3_trace_rates.csv':                  df_trace_rates,   # Step 3: bid_rate/throttle_rate/vbt_rate by format
    'kakao_s3_trace_bid_result_breakdown.csv':   df_trace_eval,    # Step 3: bid_result distribution by format
    'kakao_s3b_bid_floor.csv':                   df_floor,         # Step 3b: bid floor percentiles by format
}

for fname, df in exports.items():
    path = os.path.join(DATA_DIR, fname)
    df.to_csv(path, index=False)
    print(f'✅ {fname} ({len(df)} rows) → {path}')


✅ kakao_s1_funnel_fact_supply.csv (3 rows) → /Users/haewon.yum/Documents/Queries/Measurement/VT/data/kakao_s1_funnel_fact_supply.csv
✅ kakao_s3_trace_rates.csv (2 rows) → /Users/haewon.yum/Documents/Queries/Measurement/VT/data/kakao_s3_trace_rates.csv
✅ kakao_s3_trace_bid_result_breakdown.csv (33 rows) → /Users/haewon.yum/Documents/Queries/Measurement/VT/data/kakao_s3_trace_bid_result_breakdown.csv
✅ kakao_s3b_bid_floor.csv (2 rows) → /Users/haewon.yum/Documents/Queries/Measurement/VT/data/kakao_s3b_bid_floor.csv
